In [2]:
import pandas as pd
import openpyxl

# Mapping dei nomi dei fogli
SHEET_NAME_MAP = {
    '10001100023100042100130v1': 'digital_rapporti',
    '10001100023100042100130v2': 'digital_frodi',
    '10001100023100042100130v3': 'digital_raisin',
    '10001100023100042100129': 'bancassurance',
    '10001100023100036v1': 'monetica_bonifici_banca',
    '10001100023100036v2': 'monetica_cassa',
    '10001100023100036v3': 'monetica_cassette',
    '10001100023100036v4': 'monetica_bonifici_estero',
    '10001100023100034': 'ops_aml',
    '10001100023100044': 'antifrode',
    '10001100023000001': 'anagrafe',
    '10004100067100079': 'perfezionamenti',
    '10004100067100080v1': 'factoring_cedenti',
    '10004100067100080v2': 'factoring_debitori',
    '10004100052': 'credito',
    '10004100051': 'credito_speciale'
}

# Colonne da mantenere per ogni foglio
COLUMNS_TO_KEEP = {
    'credito': [
        'des_tipo_istruttoria',
        'des_stato_istruttoria',
        'nro_giorni_lavorazione',
        'nro_giorni_coda',
        'des_ndg_operatore_lavorazione',
        'des_organo_delib',
        'des_scopo_pratica',
        'des_tipo_delibera',
        'dta_delibera',
        'dta_istruttoria',
    ],
    'credito_speciale': [
        'des_tipo_istruttoria',
        'des_stato_istruttoria',
        'nro_giorni_lavorazione',
        'nro_giorni_coda',
        'des_ndg_operatore_lavorazione',
        'des_organo_delib',
        'des_scopo_pratica',
        'des_tipo_delibera',
        'dta_delibera',
        'dta_istruttoria',
    ],
    'perfezionamenti': [
        'des_business_unit',
        'dta_operativa',
        'dta_delibera',
    ],
    'factoring_cedenti': [
        'ndg',
        'business unit',
        'filiale',
        'gestore',
        'segnalatore',
        'data prima stipula',
        'accordato',
        'impiego',
        'turnover anno corrente',
    ],
    'factoring_debitori': [
        'ndg_debitore',
        'data_delibera',
        'accordato_pro_solvendo',
        'accordato_pro_soluto',
        'descrizione_prodotto',
    ],
    'anagrafe': [
        'des_business_unit',
        'dta_censimento',
        'des_natura_giuridica',
        'des_status_generic',
    ],
    'antifrode': [
        'conteggio',
        'classificazione',
        'cluster_frode',
        'mese',
    ],
    'ops_aml': [
        'fascia_rischio',
        'data_uscita',
        'data_inserimento',
        'data_scadenza_adv',
        'ndg',
        'business_unit',
        'cluster',
        'workflow',
        'tipo_verifica',
    ],
    'bancassurance': [
        'data_ordine',
        'descrizione_stato',
        'tot_generale_euro',
    ],
    'digital_rapporti': [
        'dta_rapporto_apert',
        'des_categoria_rapporto',
        'dta_rapporto_estinzione',
    ],
    'digital_frodi': [
        'Campo personalizzato (Data operazione (FR))',
        'Campo personalizzato (Cluster Frode Banca)',
    ],
    'digital_raisin': [
        'data_inserimento',
    ],
    'monetica_bonifici_banca': [
        'data_valuta_fissa_al_beneficiario',
        'data_regolamento',
        'importo_bonifico',
    ],
    'monetica_cassa': [
        'd_data_cont',
        'tg04_causale1',
        'descrizione_filiale',
    ],
    'monetica_cassette': [
        'dta_rapporto_apert',
        'des_business_unit',
    ],
    'monetica_bonifici_estero': [
        'data_inserimento',
        'importo_bonifico',
        'paese_ord',
        'paese_beneficiario',
        'data_inserimento',
        'filiale',
    ],
}


def get_sheet_names_from_excel(excel_path: str) -> list:
    """Estrae i nomi dei fogli"""
    xl_file = pd.ExcelFile(excel_path)
    return xl_file.sheet_names


def get_sheet_name_key(original_name: str) -> str:
    """Mappa il nome originale del foglio alla chiave"""
    for key, value in SHEET_NAME_MAP.items():
        if key in original_name:
            return value
    return None


def filter_excel(excel_input: str, excel_output: str):
    """
    Legge l'Excel.

    - Il primo foglio viene lasciato AS-IS e non viene processato.
    - Il secondo foglio viene processato ma mantiene TUTTE le colonne.
    - Dal terzo foglio in poi vengono filtrate le colonne secondo COLUMNS_TO_KEEP.
    """
    
    print("=" * 80)
    print("🚀 INIZIO ELABORAZIONE EXCEL")
    print("=" * 80 + "\n")
    
    # Estrae i nomi dei fogli
    try:
        sheet_names = get_sheet_names_from_excel(excel_input)
    except FileNotFoundError:
        print(f"❌ File non trovato: {excel_input}")
        raise
    
    print(f"📊 File caricato: {len(sheet_names)} fogli trovati")
    print(f"Fogli: {sheet_names}\n")
    
    # Crea il writer per il file output
    with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
        
        # =====================================================================
        # PRIMO FOGLIO: LASCIATO AS-IS
        # =====================================================================
        print("=" * 80)
        print("📄 PRIMO FOGLIO (NON PROCESSATO)")
        print("=" * 80 + "\n")
        
        if len(sheet_names) > 0:
            try:
                df = pd.read_excel(
                    excel_input,
                    sheet_name=sheet_names[0],
                    dtype=str,
                    keep_default_na=False
                )
                
                df.to_excel(
                    writer,
                    sheet_name=sheet_names[0],
                    index=False
                )
                
                print(f"✅ Foglio '{sheet_names[0]}' copiato AS-IS")
                print(f"   ├─ Righe: {len(df):,}")
                print(f"   └─ Colonne: {len(df.columns)}\n")
                
            except Exception as e:
                print(
                    f"❌ Errore lettura foglio "
                    f"'{sheet_names[0]}': {str(e)}\n"
                )
        
        # =====================================================================
        # DAL SECONDO FOGLIO IN POI: PROCESSAMENTO
        # =====================================================================
        print("=" * 80)
        print("🔍 FOGLI PROCESSATI DAL SECONDO IN POI")
        print("=" * 80 + "\n")
        
        processed_count = 0
        skipped_count = 0
        
        for idx, original_sheet_name in enumerate(
            sheet_names[1:],
            start=1
        ):
            
            try:
                # Legge il foglio
                df = pd.read_excel(
                    excel_input,
                    sheet_name=original_sheet_name
                )
                
            except Exception as e:
                print(
                    f"❌ [{idx}] '{original_sheet_name}': "
                    f"errore lettura - {str(e)}"
                )
                skipped_count += 1
                continue
            
            # =================================================================
            # SECONDO FOGLIO: PROCESSATO MA SENZA DROP DI COLONNE
            # =================================================================
            if idx == 1:
                
                # Mappa comunque il nome del foglio
                sheet_key = get_sheet_name_key(original_sheet_name)
                
                # Mantiene tutte le colonne
                df_filtered = df.copy()
                
                # Mantiene la logica speciale di factoring_debitori
                # anche se il secondo foglio dovesse essere questo.
                if sheet_key == 'factoring_debitori':
                    
                    if (
                        'accordato_pro_solvendo' in df_filtered.columns
                        and
                        'accordato_pro_soluto' in df_filtered.columns
                    ):
                        
                        initial_rows = len(df_filtered)
                        
                        df_filtered = df_filtered[
                            ~(
                                (
                                    pd.to_numeric(
                                        df_filtered[
                                            'accordato_pro_solvendo'
                                        ],
                                        errors='coerce'
                                    ).fillna(0) == 0
                                )
                                &
                                (
                                    pd.to_numeric(
                                        df_filtered[
                                            'accordato_pro_soluto'
                                        ],
                                        errors='coerce'
                                    ).fillna(0) == 0
                                )
                            )
                        ]
                        
                        dropped_rows = (
                            initial_rows - len(df_filtered)
                        )
                        
                        print(
                            f"   ├─ 🗑️  Righe droppe "
                            f"(entrambi zero): {dropped_rows}"
                        )
                
                # Scrive il secondo foglio mantenendo TUTTE le colonne
                df_filtered.to_excel(
                    writer,
                    sheet_name=original_sheet_name,
                    index=False
                )
                
                processed_count += 1
                
                print(
                    f"✅ [{idx}] '{original_sheet_name}'"
                    f" → SECONDO FOGLIO"
                )
                print(f"   ├─ Righe: {len(df_filtered):,}")
                print(f"   ├─ Colonne originali: {len(df.columns)}")
                print(
                    f"   ├─ Colonne mantenute: "
                    f"{len(df_filtered.columns)}"
                )
                print("   └─ Colonne droppate: 0\n")
                
                # Passa direttamente al foglio successivo
                continue
            
            # =================================================================
            # DAL TERZO FOGLIO: PROCESSAMENTO NORMALE CON FILTRO
            # =================================================================
            
            # Mappa il nome del foglio
            sheet_key = get_sheet_name_key(original_sheet_name)
            
            if sheet_key is None:
                print(
                    f"⚠️  [{idx}] '{original_sheet_name}': "
                    f"nessuna mappatura trovata - SKIPPED"
                )
                skipped_count += 1
                continue
            
            if sheet_key not in COLUMNS_TO_KEEP:
                print(
                    f"⚠️  [{idx}] '{original_sheet_name}' "
                    f"({sheet_key}): nessuna configurazione - SKIPPED"
                )
                skipped_count += 1
                continue
            
            # Filtra le colonne disponibili
            cols_to_keep = [
                col
                for col in COLUMNS_TO_KEEP[sheet_key]
                if col in df.columns
            ]
            
            if not cols_to_keep:
                print(
                    f"❌ [{idx}] '{original_sheet_name}' "
                    f"({sheet_key}): NESSUNA colonna trovata!"
                )
                print(
                    f"   Colonne richieste: "
                    f"{COLUMNS_TO_KEEP[sheet_key]}"
                )
                print(
                    f"   Colonne disponibili: "
                    f"{list(df.columns)}\n"
                )
                skipped_count += 1
                continue
            
            # Colonne mancanti
            missing_cols = [
                col
                for col in COLUMNS_TO_KEEP[sheet_key]
                if col not in df.columns
            ]
            
            # Seleziona SOLO le colonne da mantenere
            df_filtered = df[cols_to_keep]
            
            # =================================================================
            # LOGICA SPECIALE FACTORING DEBITORI
            # =================================================================
            if sheet_key == 'factoring_debitori':
                
                initial_rows = len(df_filtered)
                
                df_filtered = df_filtered[
                    ~(
                        (
                            pd.to_numeric(
                                df_filtered[
                                    'accordato_pro_solvendo'
                                ],
                                errors='coerce'
                            ).fillna(0) == 0
                        )
                        &
                        (
                            pd.to_numeric(
                                df_filtered[
                                    'accordato_pro_soluto'
                                ],
                                errors='coerce'
                            ).fillna(0) == 0
                        )
                    )
                ]
                
                dropped_rows = (
                    initial_rows - len(df_filtered)
                )
                
                print(
                    f"   ├─ 🗑️  Righe droppe "
                    f"(entrambi zero): {dropped_rows}"
                )
            
            # Scrive il foglio filtrato con il nome ORIGINALE
            df_filtered.to_excel(
                writer,
                sheet_name=original_sheet_name,
                index=False
            )
            
            processed_count += 1
            
            print(
                f"✅ [{idx}] '{original_sheet_name}' "
                f"→ {sheet_key}"
            )
            print(f"   ├─ Righe: {len(df_filtered):,}")
            print(
                f"   ├─ Colonne originali: "
                f"{len(df.columns)}"
            )
            print(
                f"   ├─ Colonne mantenute: "
                f"{len(cols_to_keep)}"
            )
            print(
                f"   ├─ Colonne droppate: "
                f"{len(df.columns) - len(cols_to_keep)}"
            )
            
            if missing_cols:
                print(
                    f"   ├─ ⚠️  Colonne mancanti: "
                    f"{missing_cols}"
                )
            
            print(
                f"   └─ Colonne: {cols_to_keep}\n"
            )
    
    # =====================================================================
    # RIEPILOGO FINALE
    # =====================================================================
    print("=" * 80)
    print("📊 RIEPILOGO ELABORAZIONE")
    print("=" * 80 + "\n")
    
    print(f"✅ Fogli processati: {processed_count}")
    print(f"⚠️  Fogli skipped: {skipped_count}")
    print(f"📄 File output: {excel_output}\n")
    
    print("=" * 80)
    print("✅ ELABORAZIONE COMPLETATA!")
    print("=" * 80)


def main():
    """Funzione principale"""
    
    # ========== CONFIGURA QUI ==========
    excel_input = 'database_v1.xlsx'
    excel_output = 'database_v1_filtrato.xlsx'
    # ===================================
    
    try:
        filter_excel(
            excel_input,
            excel_output
        )
        
    except FileNotFoundError as e:
        print(f"❌ File non trovato: {e}")
        
    except Exception as e:
        print(f"❌ Errore inaspettato: {str(e)}")
        import traceback
        traceback.print_exc()


if __name__ == '__main__':
    main()

🚀 INIZIO ELABORAZIONE EXCEL

📊 File caricato: 18 fogli trovati
Fogli: ['Dimensionamento', 'Task', '10001100023000001', '10001100023100044', '10004100052', '10004100051', '10001100023100034', '10001100023100042100130v1', '10001100023100042100130v2', '10001100023100042100130v3', '10001100023100042100129', '10004100067100079', '10001100023100036v1', '10001100023100036v2', '10001100023100036v3', '10001100023100036v4', '10004100067100080v1', '10004100067100080v2']

📄 PRIMO FOGLIO (NON PROCESSATO)

✅ Foglio 'Dimensionamento' copiato AS-IS
   ├─ Righe: 18
   └─ Colonne: 12

🔍 FOGLI PROCESSATI DAL SECONDO IN POI

✅ [1] 'Task' → SECONDO FOGLIO
   ├─ Righe: 1,040
   ├─ Colonne originali: 20
   ├─ Colonne mantenute: 20
   └─ Colonne droppate: 0

✅ [2] '10001100023000001' → anagrafe
   ├─ Righe: 10,978
   ├─ Colonne originali: 45
   ├─ Colonne mantenute: 4
   ├─ Colonne droppate: 41
   └─ Colonne: ['des_business_unit', 'dta_censimento', 'des_natura_giuridica', 'des_status_generic']

✅ [3] '1000110